#### FCNN1

Implementation was supposed to be based on the article below
Angiulli, Fabrizio. *"Fast condensed nearest neighbor rule."* Proceedings of the 22nd international conference on Machine learning. 2005. [link](https://icml.cc/Conferences/2005/proceedings/papers/004_Fast_Angiulli.pdf)

In [2]:
# Unfortunately not working and useless

In [4]:
# from sklearn.cluster import KMeans
# from sklearn.metrics.pairwise import euclidean_distances


# class FastCondensedNearestNeighbor:
#     def __init__(
#         self,
#         *,
#         n_neighbors=1,
#         metric_distances=euclidean_distances,
#         random_state=0,
#         metric="euclidean",
#         n_jobs=-1,
#     ):
#         self.n_neighbors = n_neighbors
#         self.metric_distances = metric_distances
#         self.random_state = random_state
#         self.metric = metric
#         self.n_jobs = n_jobs
#         self.iters = []
#         self.grabbag = []

#     def update_iter_stats(self, it, len_idxs):
#         self.iters.append(it)
#         self.grabbag.append(len_idxs)

#     def fit(self, X, y):
#         self.knn = KNeighborsClassifier(
#             n_neighbors=self.n_neighbors,
#             metric=self.metric,
#             n_jobs=self.n_jobs,
#         )
#         self.rng = np.random.default_rng(self.random_state)
#         self.original_samples_no = X.shape[0]
#         return self

#     def get_class_centroids(self, X, y):
#         coords_iloc_idxs, labels = [], []

#         kmeans = KMeans(
#             n_clusters=y.nunique(), random_state=0, n_init="auto"
#         ).fit(X)
#         centroids = kmeans.cluster_centers_

#         for class_centroid in centroids:
#             _, ind = self.knn.kneighbors(class_centroid.reshape(1, -1))
#             ind = ind.flatten()[0]

#             label = kmeans.labels_[ind]
#             control_label = y.iloc[ind]
#             if label != control_label:
#                 raise Exception("
#                                 get_class_centroids: labels mismatch!
#                                 ")
#                 print("Labels mismatched")

#             coords_iloc_idxs.append(ind)
#             labels.append(label)

#         return coords_iloc_idxs, labels

#     def dist(self, x, y):
#         return self.metric_distances(x, y)

#     def transform(self, X_train, y_train) -> (pd.DataFrame, pd.Series):
#         # Store bin
#         X_store = pd.DataFrame()
#         y_store = pd.Series()

#         # Keep the original dfs safe
#         X = X_train.copy(deep=True)
#         y = y_train.copy(deep=True)

#         # Training set indexes
#         idxs = np.array(X_train.index)
#         self.update_iter_stats(0, len(idxs))

#         # Samples to remove
#         grabbag_idx = []

#         # Nearest and Representative
#         nearest = {}
#         rep = {}

#         # Delta S
#         delta_iloc_idxs = []
#         # S idxs
#         s_loc_idxs = []

#         # Fit knn on training dataset
#         # Used to caluclate centroids
#         self.knn.fit(X, y)

#         # Initial centroids calculation
#         centroid_iloc_idxs, labels = self.get_class_centroids(X, y)
#         # Place them in delta S
#         for centroid in centroid_iloc_idxs:
#             print(centroid)
#             delta_iloc_idxs.append(centroid)

#         it = 0
#         while len(delta_iloc_idxs) > 0:
#             # S += delta S
#             print(f"delta_iloc_idxs: {delta_iloc_idxs}")
#             for i, iloc_idx in enumerate(delta_iloc_idxs):
#                 X_store = X_store.append(
#                     X_train.iloc[iloc_idx].copy(deep=True)
#                 )
#                 y_store = pd.concat(
#                     [
#                         y_store,
#                         pd.Series(
#                             y_train.iloc[iloc_idx].copy(), index=[iloc_idx]
#                         ),
#                     ]
#                 )

#                 loc_idx = X_train.iloc[iloc_idx].name
#                 X = X.drop(loc_idx)
#                 y = y.drop(loc_idx)

#                 s_loc_idxs.append(loc_idx)
#                 grabbag_idx.append(i)

#             # Update sets (grabbag vs store)
#             idxs = np.delete(idxs, grabbag_idx)
#             # self.update_iter_stats(it := it + 1, len(idxs))

#             # Erase representatives
#             rep = {}

#             for i, loc_idx in enumerate(idxs):
#                 q = loc_idx
#                 for delta_iloc_idx in delta_iloc_idxs:
#                     p = X_train.iloc[delta_iloc_idx].name
#                     nearest[q] = p

#                     if i == 0:
#                         nearest[q] = p
#                         print("1")
#                         continue

#                     if q in nearest.keys():
#                         # print("Q in nearest keys")
#                         q_sample = X_train.loc[q].values.reshape(1, -1)
#                         nearest_q_sample = X_train.loc[
#                             nearest[q]
#                         ].values.reshape(1, -1)
#                         p_sample = X_train.loc[p].values.reshape(1, -1)
#                         if (self.dist(nearest_q_sample, q_sample)) <= (
#                             self.dist(p_sample, q_sample)
#                         ):
#                             nearest[q] = p
#                             # print("Satisfied")

#                 nearest_defined = q in nearest.keys()
#                 rep_nearest_q_defined = (
#                     nearest_defined and nearest[q] in rep.keys()
#                 )
#                 safety_cond = nearest_defined and rep_nearest_q_defined

#                 lables_different = (
#                     y_train.loc[q] != y_train.loc[nearest[q]]
#                     if nearest_defined
#                     else False
#                 )

#                 if nearest_defined:
#                     q_sample = X_train.loc[q].values.reshape(1, -1)
#                     nearest_q_sample = 
#                                 X_train.loc[nearest[q]].values.reshape(1, -1)
#                     rep_nearest_q_sample = X_train.loc[
#                         nearest[q]
#                     ].values.reshape(1, -1)

#                     triangle_cond = (
#                         self.dist(nearest_q_sample, q_sample)
#                     ) <= (self.dist(nearest_q_sample, rep_nearest_q_sample))

#                 print(f"
#                       Triangle cond: {triangle_cond},
#                       nearest_defined: {nearest_defined},
#                       safety_cond: {safety_cond}
#                       ")

#                 if safety_cond and labels_different and triangle_condition:
#                     rep[nearest[q]] = q

#                 # Not sure about it
#                 if i == 0 and safety_cond and labels_different:
#                     rep[nearest[q]] = q

#             delta_iloc_idxs = []
#             print(s_loc_idxs)
#             for loc_idx in s_loc_idxs:
#                 rep_p_defined = loc_idx in rep.keys()
#                 if rep_p_defined:
#                     print("Rep defined")
#                     delta_iloc_idxs.append(rep[loc_idx])

#             idxs = np.delete(idxs, grabbag_idx)
#             self.update_iter_stats(it := it + 1, len(idxs))

#         return X_store, y_store